# Caso 07: Inspecciones sanitarias de Chicago

## Integrantes:
- Graciela Gómez
- Benjamín Gutiérrez
- Juan Pablo Mondaca
- Benjamín Vargas

# 1. Carga base de datos + filtrado de la población

Todos los comentarios respecto a la decisión del filtrado se encuentran en la documentación.

In [19]:
import pandas as pd
import numpy as np

# Cambia el nombre por el de tu archivo
insp = pd.read_csv("datos.csv", dtype=str)

print(insp.shape)
insp.head()


(104468, 17)


,inspection_id,dba_name,aka_name,license_,facility_type,risk,address,city,state,zip,inspection_date,inspection_type,results,violations,latitude,longitude,location
0,2243957,TWO MILE COFFEE BAR,TWO MILE COFEE BAR,2641611,Restaurant,Risk 2 (Medium),1766 W 95TH ST,CHICAGO,IL,60643,2019-01-02T00:00:00.000,License,Not Ready,NaN,41.72126016238349,-87.66725709711507,"(41.72126016238349, -87.66725709711507)"
1,2243958,FAMILY DOLLART #32124,FAMILY DOLLAR,2630979,Grocery Store,Risk 3 (Low),1633 W MONTEREY AVE,CHICAGO,IL,60643,2019-01-02T00:00:00.000,License,Not Ready,NaN,41.689431926833365,-87.66301447393185,"(41.689431926833365, -87.66301447393185)"
2,2243959,CARMELO'S TACO PLACE,CARMELO'S TACO PLACE,2617376,Restaurant,Risk 1 (High),2746 W 59TH ST,CHICAGO,IL,60629,2019-01-02T00:00:00.000,License,Not Ready,NaN,41.78652776994852,-87.69303014768134,"(41.78652776994852, -87.69303014768134)"
3,2243960,AVONDALE TAP,AVONDALE TAP,2368771,Restaurant,Risk 1 (High),3634 W BELMONT AVE,CHICAGO,IL,60618,2019-01-02T00:00:00.000,Canvass,No Entry,NaN,41.939307518988336,-87.71873540202452,"(41.939307518988336, -87.71873540202452)"
4,2243961,MR. POLLO,MR. POLLO,2542966,Restaurant,Risk 1 (High),3026 W BELMONT AVE,CHICAGO,IL,60618,2019-01-02T00:00:00.000,Canvass,Pass w/ Conditions,"3. MANAGEMENT, FOOD EMPLOYEE AND CONDITIONAL E...",41.93946745056518,-87.70381061827938,"(41.93946745056518, -87.70381061827938)"


In [20]:
# Identificador de establecimiento: licencia válida
insp["licencia_limpia"] = (
    insp["license_"].fillna("").str.strip()
) 
insp["licencia_valida"] = insp["license_"].notna() & (insp["license_"] != "0")
print("Filas con licencia 0 o vacía:", (~insp["licencia_valida"]).sum())

# Fecha como fecha
insp["inspection_date"] = pd.to_datetime(insp["inspection_date"])

# Filas objetivo: solo Canvass exacto (sin reinspecciones)
tipo = insp["inspection_type"].str.strip().str.lower()
canvass = insp[tipo == "canvass"].copy()

print("Inspecciones Canvass:", len(canvass))              # 52.675
print("Licencias distintas:", canvass["license_"].nunique())
print("Canvass sin licencia válida:", (~canvass["licencia_valida"]).sum())



Filas con licencia 0 o vacía: 252
Inspecciones Canvass: 52675
Licencias distintas: 18081
Canvass sin licencia válida: 127


In [ ]:
#Normalización de las direcciones
insp["direccion_normalizada"] = (insp["address"]
.fillna("")
.str.upper() #Mayúsculas
.str.strip() #Espacios en blanco
.str.replace(r"[.,;]", "", regex = True) #puntos y puntocoma en espacios
.str.replace(r"\s+"," ",regex = True) #espacios dobles en uno
.str.strip()
)
insp["direccion_valida"] = ~insp["direccion_normalizada"].isin(["","0", "UNKOWN","N/A"])


#identificación id local
identificable = (insp["licencia_valida"] & insp["direccion_valida"])
insp["id_local"] = None
insp.loc[identificable, "id_local"] = (insp["licencia_limpia"]+ "|" + insp["direccion_normalizada"])

print("Locales distintos:", insp["id_local"].nunique())
print("Inspecciones sin identificación:", insp["id_local"].isna().sum())

licencias_por_direccion = (
    insp[insp["licencia_valida"] & insp["direccion_valida"]]
    .groupby("direccion_normalizada")["licencia_limpia"]
    .nunique()
)

print("Direcciones con distintas licencias:",
      (licencias_por_direccion > 1).sum())
print(insp.columns)

Locales distintos: 23705
Inspecciones sin identificación: 252
Direcciones con distintas licencias: 4729
Index(['inspection_id', 'dba_name', 'aka_name', 'license_', 'facility_type',
       'risk', 'address', 'city', 'state', 'zip', 'inspection_date',
       'inspection_type', 'results', 'violations', 'latitude', 'longitude',
       'location', 'licencia_limpia', 'licencia_valida',
       'direccion_normalizada', 'direccion_valida', 'id_local'],
      dtype='str')
0   2019-01-02
1   2019-01-02
2   2019-01-02
3   2019-01-02
4   2019-01-02
Name: inspection_date, dtype: datetime64[us]


In [29]:
#Construcción de antecedentes en un dataframe

#filtrar inspecciones canvass
filtro = (
    (insp["inspection_type"].str.strip().str.lower() == "canvass")&(insp["id_local"].notna())
)

#dtaframe
df_historial = pd.crosstab(
    insp.loc[filtro, "id_local"],
    insp.loc[filtro, "results"].str.strip().str.lower()
)

df_historial = df_historial.reindex( 
    columns = ["pass","pass w/ conditions", "fail"],
    fill_value = 0
).reset_index()

print(df_historial.head())

results                       id_local  pass  pass w/ conditions  fail
0           1000774|3655 N WESTERN AVE     1                   4     0
1         1000894|1300 S LAKE SHORE DR     1                   1     0
2                1000|600 S CICERO AVE     0                   1     0
3               10166|201 E ONTARIO ST     4                   1     0
4        10175|3750-3752 N CENTRAL AVE     0                   0     0


In [ ]:
#antecedentes historicos
#copia
historial = insp.copy()

#ordenamos dentro de cada local desde la inspección más antigua
#a la más reciente. (todos no solo canvass)
historial["fecha"] = historial["inspection_date"]
historial = historial.sort_values(["id_local","fecha"])

#verificación de que la cosa funciona 
id_ejemplo = historial["id_local"].value_counts().index[0]
historial.loc[
    historial["id_local"] == id_ejemplo,
    ["id_local", "fecha", "inspection_type", "results"]
].head(10)

#variables de conteo
historial["resultado_limpio"] = ( historial["results"].fillna("")
.str.strip()
.str.lower())

historial["pass"] = (historial["resultado_limpio"] == "pass").astype(int)

historial["fail"] = (historial["resultado_limpio"] == "fail").astype(int)

historial["conditions"] = (historial["resultado_limpio"] == "pass w/ conditions").astype(int)


historial = historial[["inspection_id", "id_local", "fecha",
"inspection_type", "pass", "fail", "conditions"]].copy()

print(historial.head(10))

      inspection_id                         id_local      fecha  \
8056        2291046  1000572|7530 S STONY ISLAND AVE 2019-06-06   
23344       2366339  1000572|7530 S STONY ISLAND AVE 2020-03-26   
24626       2369727  1000572|7530 S STONY ISLAND AVE 2020-04-22   
31706       2453722  1000572|7530 S STONY ISLAND AVE 2020-10-23   
48269       2544302  1000572|7530 S STONY ISLAND AVE 2021-11-17   
48900       2544934  1000572|7530 S STONY ISLAND AVE 2021-12-02   
54043       2554105  1000572|7530 S STONY ISLAND AVE 2022-04-08   
60659       2562760  1000572|7530 S STONY ISLAND AVE 2022-08-25   
64584       2566712  1000572|7530 S STONY ISLAND AVE 2022-11-07   
69283       2571462  1000572|7530 S STONY ISLAND AVE 2023-02-21   

               inspection_type  pass  fail  conditions  
8056                 Complaint     0     0           1  
23344                Complaint     0     0           1  
24626     Short Form Complaint     0     0           1  
31706     Short Form Complaint    